# Lab 13 solution, notebook 2: Helena's rep and the first-reply time, and whether the agent was needed

The worked answers to the two challenge tasks of `13b_agentic_rag_at_the_desk.ipynb`. The setup repeats what the
tasks need from the notebook (the files, the search, `answer_once`, the tool, the system message, `run_loop` and
`borrowed_words`), so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import json
import re
import numpy as np
from openai import OpenAI, omit

client = OpenAI(timeout=60)
EMBEDDING_MODEL = settings["EMBEDDING_MODEL"]

# The shop's 36 files, embedded in one request: row i of D is the vector of names[i]
FILES = {path.name: path.read_text() for path in sorted((ROOT / "data" / "corpus").glob("*.md"))}
names = list(FILES)
response = client.embeddings.create(model=EMBEDDING_MODEL, input=list(FILES.values()))
D = np.array([item.embedding for item in response.data])


def nearest(query, k):
    """The k files nearest to the query, as (file name, score) pairs, nearest first."""
    q = np.array(client.embeddings.create(model=EMBEDDING_MODEL, input=query).data[0].embedding)
    scores = D @ q
    return [(names[i], float(scores[i])) for i in np.argsort(-scores)[:k]]


def search(query, k=2):
    """The names of the k nearest files, and one text with each file's name in brackets, then its text."""
    found = [name for name, _ in nearest(query, k)]
    return found, "\n\n".join(f"[{name}]\n{FILES[name]}" for name in found)

In [3]:
ANSWER_RULES = "Answer only from the files. If they do not say, say you do not know."


def answer_once(question, k):
    files, text = search(question, k)
    return client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": ANSWER_RULES},
            {"role": "user", "content": f"Files:\n\n{text}\n\nQuestion: {question}"},
        ],
        reasoning_effort=EFFORT or omit,
    )


search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the shop's policy and ticket files with a short query of a few words. Returns the two nearest files.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

SYSTEM = (
    "Answer only from the shop's files. Use the search tool to find them. "
    "Search again while the question is still open. If the files do not say, say you do not know."
)

In [4]:
def run_loop(question, max_turns=6):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    queries, usage = [], []

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=[search_tool], parallel_tool_calls=False, reasoning_effort=EFFORT or omit
        )
        usage.append(response.usage)
        reply = response.choices[0].message
        messages.append(reply)

        # The model answered: return its text, the queries and the token counts
        if not reply.tool_calls:
            return reply.content, queries, usage

        for call in reply.tool_calls:
            query = json.loads(call.function.arguments)["query"]
            files, result = search(query)
            print(f"turn {turn}: search({query!r}) -> {files}")
            queries.append((turn, query, files))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    # The turn cap stopped the loop: no answer
    return None, queries, usage


def words(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def borrowed_words(query, question, files_read):
    text_read = " ".join(FILES[name] for name in files_read)
    return sorted((words(query) - words(question)) & words(text_read))

## Task 1: Helena's rep and the first-reply time

Two facts, two files: ticket 01 names her rep, and the support-hours policy gives the first-reply time. The check
below reads the answer for both facts and lists the files the searches returned.

In [5]:
HELENA = (
    "Helena received the wrong album. Who is her assigned support representative, "
    "and how soon should support send a first reply?"
)

answer, queries, usage = run_loop(HELENA)
print()
print(answer)

files_found = {name for _, _, found in queries for name in found}
print()
print("✅" if "steve johnson" in answer.lower() else "❌", "names Steve Johnson")
print("✅" if "1 business day" in answer.lower() else "❌", "gives 1 business day")
print("ticket 01 found:", "ticket_01_helena_wrong_album.md" in files_found,
      "| support hours found:", "policy_support_hours.md" in files_found)

turn 1: search('Helena wrong album support representative first reply') -> ['ticket_01_helena_wrong_album.md', 'policy_support_hours.md']


turn 2: search('Helena assigned rep support first reply SLA') -> ['policy_support_hours.md', 'policy_support_reps.md']



Helena’s assigned representative is **Steve Johnson**. Support should send the first reply **within 1 business day**.

✅ names Steve Johnson
✅ gives 1 business day
ticket 01 found: True | support hours found: True


Two searches and both facts. The first search, written from the question's words, already returned both files that answer it: `ticket_01_helena_wrong_album.md` and `policy_support_hours.md`. Turn 2 searched again, *Helena assigned rep support first reply SLA*, and added no new fact. The answer names Steve Johnson and gives 1 business day.

## Task 2: Was the agent needed?

Where did each query's words come from? `borrowed_words` lists the words of a query that are not in the question and
are in a file returned by an earlier search.

In [6]:
for turn, query, files in queries:
    read_before = [name for t, _, found in queries if t < turn for name in found]
    print(f"turn {turn}: {query!r:55} borrowed: {borrowed_words(query, HELENA, read_before)}")

turn 1: 'Helena wrong album support representative first reply' borrowed: []
turn 2: 'Helena assigned rep support first reply SLA'           borrowed: ['rep', 'sla']


Turn 1 borrows nothing: every word of its query is in the question. Turn 2 borrows *rep* (ticket 01's "Assigned rep") and *sla* (the support-hours policy's title, "Support hours and SLA"), but both files that answer the question had already come back on turn 1. Borrowed words show that a query was written after a file was read; they do not show that the search was needed.

Now one search with the question's own words, at `k=2`:

In [7]:
print("two nearest files:", [name for name, _ in nearest(HELENA, 2)])

single = answer_once(HELENA, k=2)
print(single.choices[0].message.content)
print()
print("single search: 1 call,", single.usage.prompt_tokens, "input tokens | agent:", len(usage), "calls,",
      sum(u.prompt_tokens for u in usage), "input tokens")

two nearest files: ['ticket_01_helena_wrong_album.md', 'policy_support_hours.md']


Helena’s assigned representative is Steve Johnson. Support should send a first reply within 1 business day.

single search: 1 call, 233 input tokens | agent: 3 calls, 1244 input tokens


The two nearest files are the ticket and the support-hours policy, and one call answers both facts: 233 input tokens, against the agent's 1,244 over 3 calls. This question did not need the agent.

**Why the warped-vinyl question was different.** There, the rule for a second damaged copy was in a policy that the
question's words ranked 5th, and only ticket 06 named it. Here the question already says "support representative" and
"first reply", and the two files that answer it are the two nearest. The ticket gives an answer (Steve Johnson), not a
pointer to another file, so there is nothing to look up in a second search.